# Mean VOC and NOx Analysis

This notebook iteratively processes CSV files in a specified directory, calculates the mean VOC and mean NOx for each dataset, and outputs a summary CSV.

**Filters applied:**
- Rows with NaN values are dropped.
- Rows with 0 Ozone are dropped.
- Rows with Ozone > `ozone_cutoff` are dropped.

In [1]:
import pandas as pd
import os
import glob

In [23]:
# Configurable Parameters
target_directory = '/Users/isaacsudweeks/Desktop/Work/Improved_Isopleths/data/utah/filtered_by_clearing_index/Weekend_Split'
VOC_threshold = 900  # Define the max Ozone threshold here

In [24]:
# Get absolute path for clarity
abs_path = os.path.abspath(target_directory)
print(f"Processing files in: {abs_path}")

# Initialize a list to store the results
results = []

# Find all CSV files in the directory
file_pattern = os.path.join(abs_path, '*.csv')
csv_files = glob.glob(file_pattern)

print(f"Found {len(csv_files)} CSV files.")

for file_path in csv_files:
    try:
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        # Check for necessary columns
        required_cols = ['VOC', 'NOx', 'Ozone']
        if all(col in df.columns for col in required_cols):
            
            # 1. Drop Data with any NaN values
            df = df.dropna()
            
            # 2. Filter out zero Ozone
            df = df[df['Ozone'] != 0]
            
            # 3. Filter out high Ozone
            df = df[df['VOC'] <= VOC_threshold]
            
            if not df.empty:
                # Calculate means
                mean_voc = df['VOC'].mean()
                mean_nox = df['NOx'].mean()
                
                # Store the dataset name and the calculated means
                filename = os.path.basename(file_path)
                results.append({
                    'Dataset': filename,
                    'Mean_VOC': mean_voc,
                    'Mean_NOx': mean_nox
                })
            else:
                 print(f"Skipping {os.path.basename(file_path)}: Dataframe empty after filtering.")

        else:
            print(f"Skipping {os.path.basename(file_path)}: Missing one or more columns: {required_cols}")
            
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

Processing files in: /Users/isaacsudweeks/Desktop/Work/Improved_Isopleths/data/utah/filtered_by_clearing_index/Weekend_Split
Found 8 CSV files.


In [25]:
# Create a DataFrame from the results
results_df = pd.DataFrame(results)

# Display the DataFrame
print(results_df)

# Output result to a CSV file
output_filename = 'mean_values_summary.csv'
results_df.to_csv(target_directory + '/' + output_filename, index=False)

print(f"\nSummary saved to {os.path.abspath(output_filename)}")

                                 Dataset   Mean_VOC  Mean_NOx
0     SR_clearing_nonmax_all_weekday.csv  22.897863  3.615832
1     SR_clearing_nonmax_all_weekend.csv  22.139211  2.798618
2  SR_clearing_nonmax_summer_weekend.csv  21.220313  2.625746
3    SR_clearing_nonmax_fall_weekend.csv  22.664800  3.520000
4  SR_clearing_nonmax_spring_weekday.csv  18.887189  3.140476
5  SR_clearing_nonmax_summer_weekday.csv  24.232827  3.705694
6    SR_clearing_nonmax_fall_weekday.csv  20.124660  4.260000
7  SR_clearing_nonmax_spring_weekend.csv  19.928331  2.589744

Summary saved to /Users/isaacsudweeks/Desktop/Work/Improved_Isopleths/Notebooks/mean_values_summary.csv
